# Ollama & Local LLMs Walkthrough

Remember last class? We built all that Python scaffolding - `build_messages()`, `safe_parse_json()`, `route_prompt()` - and ended with: **"Soon, we'll plug these functions into a real LLM running on your machine."**

That moment is now.

You've installed Ollama. You've pulled a model. Now let's connect everything and see what a local LLM can actually do.

## Part 1: Setup Check

Before we do anything, let's make sure Ollama is running and you have a model ready.

In [ ]:
# Check that Ollama is installed
!ollama --version

In [ ]:
# See what models you have downloaded
!ollama list

In [ ]:
# If you don't have a model yet, pull one (this downloads ~4GB)
# Uncomment the line below if needed:
# !ollama pull llama3.2

In [ ]:
# ✏️ SET YOUR MODEL HERE
# Pick one from the list above. All cells below will use this model.
MODEL = "llama3.2"

## Part 2: Talking to Ollama from Python

Last class we used `build_messages()` to construct prompts. Now we'll actually **send** them to a model. We'll use Ollama's local HTTP API via `requests` - the same pattern used by cloud LLM APIs.

In [ ]:
import requests
import json

# Our helper from last class - now it actually does something
def build_messages(system_prompt, user_input):
    """Build a standard message list for any LLM API."""
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ]

def ask_ollama(prompt, model=MODEL):
    """Send a prompt to Ollama and get a response."""
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False},
        timeout=120
    )
    return response.json()["response"].strip()

# Your first real local LLM call!
response = ask_ollama("What is life? Answer in one sentence.")
print(response)

That response came from a model running **entirely on your machine**. No internet required. No API key. No cost.

Now let's plug in the prompt techniques from last class.

## Part 3: Prompt Techniques - Now for Real

Last class you built the prompts. You pasted them into ChatGPT or Claude. Now let's run them against your **own** model and see the difference each technique makes.

### 3.1 Zero-Shot vs Few-Shot

Give the model a task with no examples, then with examples. Compare the results.

In [ ]:
# Zero-shot: no examples, just the task
zero_shot = """Classify this movie review as Positive, Negative, or Neutral. 
Reply with ONLY the classification.

Review: "The acting was wooden and the plot made no sense, but the soundtrack was incredible."
"""

print("ZERO-SHOT:")
print(ask_ollama(zero_shot))
print()

In [ ]:
# Few-shot: give examples first so the model learns the pattern
few_shot = """Classify each movie review as Positive, Negative, or Neutral.
Reply with ONLY the classification.

Review: "Absolutely loved every minute. A masterpiece."
Classification: Positive

Review: "Worst movie I've seen this year. Total waste of time."
Classification: Negative

Review: "It was okay. Nothing special but not terrible either."
Classification: Neutral

Review: "The acting was wooden and the plot made no sense, but the soundtrack was incredible."
Classification:"""

print("FEW-SHOT:")
print(ask_ollama(few_shot))
print()
print("Compare: Did few-shot give a more consistent, concise answer?")

### 3.2 Chain-of-Thought

Ask the model to solve a logic puzzle - first without reasoning, then with "think step by step."

In [ ]:
# Without chain-of-thought
puzzle = "A farmer has 15 sheep. All but 8 die. How many sheep are left?"

print("WITHOUT chain-of-thought:")
print(ask_ollama(puzzle))
print()

In [ ]:
# With chain-of-thought
puzzle_cot = """A farmer has 15 sheep. All but 8 die. How many sheep are left?

Think step by step before giving your final answer."""

print("WITH chain-of-thought:")
print(ask_ollama(puzzle_cot))
print()
print("Did the step-by-step reasoning help the model get the tricky wording right?")

### 3.3 System Prompt Personas

Remember `build_messages()` from last class? We used it to set up different personas. Now let's see those personas actually come to life.

In [ ]:
# With Ollama CLI, we combine system + user into one prompt
# (The API version uses separate fields - we'll get there next class)

def ask_with_persona(system_prompt, user_input, model=MODEL):
    """Combine a system prompt and user input, then send to Ollama."""
    full_prompt = f"""SYSTEM: {system_prompt}

USER: {user_input}"""
    return ask_ollama(full_prompt, model)

question = "Explain what a neural network is."

personas = {
    "Pirate Captain": "You are a pirate captain. Answer everything in pirate speak. Say 'arrr' frequently.",
    "5-Year-Old": "You are explaining things to a 5-year-old. Use very simple words and fun comparisons and use 'and then' a lot.",
    "Shakespeare": "You are William Shakespeare. Respond in Elizabethan English with dramatic flair."
}

for name, system in personas.items():
    print(f"\n{'='*50}")
    print(f"PERSONA: {name}")
    print(f"{'='*50}")
    response = ask_with_persona(system, question)
    print(response)

### 3.4 Structured Output

Last class we practiced parsing JSON from simulated responses. Let's see if the local model can actually produce clean JSON.

In [ ]:
# Ask the model for structured JSON output
json_prompt = """List 3 fictional planets with the following details for each:
- name
- climate
- population (as a number)

Return ONLY valid JSON as a list of objects. No other text."""

print("Asking for JSON output...")
response = ask_ollama(json_prompt)
print(response)

In [ ]:
# Now let's use safe_parse_json from last class to handle it

def safe_parse_json(response_text):
    """Try to parse JSON from an LLM response. Handle common issues."""
    cleaned = response_text.strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    if cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()
    
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"JSON parse failed: {e}")
        return None

# Try parsing the model's response
planets = safe_parse_json(response)

if planets:
    print("Successfully parsed! Here are your planets:\n")
    for p in planets:
        print(f"  {p['name']} - {p['climate']}, population: {p['population']:,}")
else:
    print("The model didn't return clean JSON. This happens!")
    print("Try running the cell above again - local models aren't always consistent.")

### 3.5 Temperature

Temperature controls how "creative" vs "predictable" the model is. Low temperature (closer to 0) makes the model pick the most likely next word every time. High temperature (closer to 2) makes it explore less likely options, producing more varied and surprising output.

- **temperature=0.0** — deterministic, always picks the top choice. Great for factual Q&A, classification, math.
- **temperature=0.7** — balanced (this is the default for most models). Good for general use.
- **temperature=1.5+** — wild and creative. Good for brainstorming, poetry, humor.

In [ ]:
def ask_ollama_with_temp(prompt, model=MODEL, temperature=0.7):
    """Send a prompt to Ollama with a specific temperature."""
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False, "options": {"temperature": temperature}},
        timeout=120
    )
    return response.json()["response"].strip()

prompt = "Invent a name for a new ice cream flavor."

print("TEMPERATURE 0.0 (run 3 times - should be identical):")
for i in range(3):
    print(f"  {i+1}. {ask_ollama_with_temp(prompt, temperature=0.0)}")

print("\nTEMPERATURE 1.5 (run 3 times - should vary):")
for i in range(3):
    print(f"  {i+1}. {ask_ollama_with_temp(prompt, temperature=1.5)}")

### 3.6 The Prompt Router - Live

Last class we built `route_prompt()` but could only simulate responses. Now let's connect it for real.

In [ ]:
# The prompt router from last class - now connected to Ollama

SYSTEM_PROMPTS = {
    "support": "You are a customer support agent. Be polite and helpful. Keep responses under 3 sentences.",
    "analyst": "You are a data analyst. Be precise. Show your reasoning. Return structured data when possible.",
    "writer": "You are a professional copywriter. Write in a clear, engaging tone. Keep it punchy."
}

def route_and_ask(task_type, user_input):
    """Route to the right persona and get a real response."""
    if task_type not in SYSTEM_PROMPTS:
        print(f"Unknown task type: '{task_type}'")
        return None
    return ask_with_persona(SYSTEM_PROMPTS[task_type], user_input)

# Same questions from last class - but now with real answers!
test_cases = [
    ("support", "I was charged twice this month. Can you help?"),
    ("analyst", "Compare these two months: January had 150 sales, February had 210 sales."),
    ("writer", "Write a tagline for a coffee shop that's open 24 hours."),
]

for task, question in test_cases:
    print(f"\n{'='*50}")
    print(f"TASK: {task.upper()}")
    print(f"INPUT: {question}")
    print(f"{'='*50}")
    print(route_and_ask(task, question))

## Part 4: Fun & Creative Scenarios

You've got a model on your machine. Let's push it and have some fun.

### 4.1 AI Storyteller

Same prompt, two completely different genres. See how the model adapts tone.

In [ ]:
setting = "a library at midnight"

horror = ask_with_persona(
    "You are a horror writer. Write exactly 3 sentences. Be creepy and atmospheric.",
    f"Write a story set in {setting}."
)

comedy = ask_with_persona(
    "You are a comedy writer. Write exactly 3 sentences. Be funny and absurd.",
    f"Write a story set in {setting}."
)

print("HORROR VERSION:")
print(horror)
print(f"\n{'='*50}\n")
print("COMEDY VERSION:")
print(comedy)

### 4.2 Recipe Inventor

Give the model random ingredients and two very different chef personas.

In [ ]:
ingredients = "peanut butter, sriracha, and leftover rice"

fancy = ask_with_persona(
    "You are a Michelin-star chef. Describe the dish elegantly. Keep it under 4 sentences.",
    f"Invent a dish using only: {ingredients}"
)

broke = ask_with_persona(
    "You are a broke college student. Be casual and funny. Keep it under 4 sentences.",
    f"Invent a dish using only: {ingredients}"
)

print("MICHELIN-STAR CHEF:")
print(fancy)
print(f"\n{'='*50}\n")
print("BROKE COLLEGE STUDENT:")
print(broke)

### 4.3 Emoji Translator

Can the model translate to emojis and back? Let's find out how much meaning survives the round trip.

In [ ]:
original = "I woke up late, spilled coffee on my shirt, but then found $20 on the ground."

# Step 1: Translate to emojis
emojis = ask_ollama(f"Translate this sentence into ONLY emojis. No words at all.\n\n\"{original}\"")
print(f"Original:   {original}")
print(f"As emojis:  {emojis}")

# Step 2: Translate back
back = ask_ollama(f"Translate these emojis back into a normal English sentence:\n\n{emojis}")
print(f"Back to English: {back}")
print(f"\nHow much meaning survived the round trip?")

### 4.4 Debate Club

Ask the model to argue both sides of a controversial (but fun) topic.

In [ ]:
topic = "pineapple on pizza"

pro = ask_with_persona(
    "You are a passionate debater. Argue IN FAVOR of the topic. Be convincing. 3-4 sentences max.",
    f"Argue FOR: {topic}"
)

con = ask_with_persona(
    "You are a passionate debater. Argue AGAINST the topic. Be convincing. 3-4 sentences max.",
    f"Argue AGAINST: {topic}"
)

print(f"TOPIC: {topic.upper()}\n")
print("FOR:")
print(pro)
print(f"\n{'='*50}\n")
print("AGAINST:")
print(con)
print("\nNotice: same model, same question, different system prompt = totally different output.")

### 4.5 Song Lyric Generator

Same topic, two genres. See how the model shifts style.

In [ ]:
song_topic = "learning to code"

country = ask_with_persona(
    "You are a country songwriter. Write one verse and one chorus. Keep it short.",
    f"Write a song about: {song_topic}"
)

rap = ask_with_persona(
    "You are a rapper. Write one verse and one chorus. Keep it short. Include some rhymes.",
    f"Write a song about: {song_topic}"
)

print("COUNTRY VERSION:")
print(country)
print(f"\n{'='*50}\n")
print("RAP VERSION:")
print(rap)

## Part 5: Putting It All Together

Combine multiple techniques in a single prompt. This is how real AI applications work.

### 5.1 Technique Combo: System Prompt + Few-Shot + JSON Output

Combine everything from Day 1 into one prompt and see if the local model can handle it.

In [ ]:
# Combining: system prompt + few-shot examples + structured JSON output
combo_prompt = """SYSTEM: You are a movie genre classifier. You are precise and consistent. 
Return ONLY valid JSON arrays.

Here is an example of your output format:

Movies: "The Shawshank Redemption", "Toy Story"
Output: [{"title": "The Shawshank Redemption", "genre": "Drama", "confidence": 0.95}, {"title": "Toy Story", "genre": "Animation/Comedy", "confidence": 0.98}]

Now classify these movies. Return a single JSON array:

Movies: "Inception", "The Notebook", "Get Out", "Finding Nemo", "The Matrix"
Output:"""

print("Sending combo prompt (system + few-shot + JSON)...\n")
response = ask_ollama(combo_prompt)
print(response)

# Try to parse it
results = safe_parse_json(response)
if results:
    print("\nParsed successfully!")
    for movie in results:
        print(f"  {movie['title']}: {movie['genre']} (confidence: {movie['confidence']:.0%})")

---

## What's Next

You now have the full loop:
- **Day 1:** Built the Python scaffolding - message formats, templates, JSON parsing, prompt routing
- **Today:** Plugged it all into a real LLM running on your machine via Ollama

**Next class:** We'll move from CLI to the **Ollama Python API** and start building real applications - calling models programmatically, handling conversations, and exploring multimodal AI (text + images).

The building blocks are in place. Now we start building.